**How Indicative Are Age and Number of Hours Played on a UBC scientist-run Minecraft Server of a Player's Subscription Status to the game-related newsletter?**

## (1) Introduction:

This project aims to use data collected from a minecraft server run by UBC scientists to answer the following broad question: “What player characteristics and behaviours are most predictive of subscribing to a game-related newsletter, and how do these features differ between various player types?” More specifically, we would like to address the following question: “Can age and number of hours played predict whether a player is subscribed to the game-related newsletter?” This will help identify whether age and number of hours are predictive of subscribing to the game-related newsletter. To help answer these questions, we utilized data from the players.csv dataset. This file contains player age, number of hours played, and newsletter subscription status. This dataset also includes 196 observations. This data was collected through a minecraft server set up by a research group in computer science at UBC. A potential problem with it is that it has the subscription status of the player stored as a logical type of data. This will need to be changed to a factor type of data in order to perform KNN classification. In addition, people may lie about their name, gender, and age. Some players also report zero hours of gameplay, but it is unclear whether it is because they are new or simply uninterested in playing. This distinction is important in the context of making predictions. For example, if 18-year-olds normally game a lot, but have few played hours because they only recently joined the study, it could skew the predicted played hours for similar age individuals. In addition, if the "experience" variable is self-reported, then designations may be subjective and unreliable as predictors. Furthermore, age in this context is technically not a quantitative variable, since it is not continuous (technically it is discrete and qualitative), as it is only composed of integer numbers. However, for the purposes of this project, we will consider age as a quantitative variable so we can use KNN classification on the dataset. We have included a small section of the players.csv dataset below to help visualize what the dataset looks like.

## (2) Methods:

## (3) Code and Results:

The players.csv file contains 196 observations, each representing a unique player. 7 variables capture the demographic information and skill level of each player. Many players report zero hours of gameplay, but it is unclear whether it is because they are new or simply uninterested in playing. This distinction is important in the context of making predictions. For example, if 18-year-olds normally game a lot, but have few played hours because they only recently joined the study, it could skew the predicted played hours for similar age individuals. In addition, if the "experience" variable is self-reported, then designations may be subjective and unreliable as predictors.

| Variable Name | Type | Description | No. of Unique Values | Mean | Standard Deviation | Min | Max | Median |
| -------- | ------- | ------- | ------- | ------- | ------- | ------- | ------- | ------- |
| experience | fct (loaded in as chr) | level of experience the player has with gaming | 5 | | | | | |
| subscribe | lgl | whether the player is subscribed to the game's associated newsletter  | 2 | | | | | |
| hashedEmail | chr | player's email encrypted as a code | 196 | | | | | |
| played_hours | dbl | no. of hours the player has played on the Minecraft server | 43 | 5.845918 |28.35734 | 0 | 223.1 | 0.1 |
| name | chr | player's first name | 196 | | | | | |
| gender | fct (loaded in as chr) | player's gender | 7 | | | | | |
| Age | int (loaded in as dbl) | player's age in years | 31 | 20.52062 | 6.174667 | 8 | 50 | 19 |

The `subscribe` variable type needs to be converted from the logical to the factor type to be used as label for KNN classification––this can be done using `fct_recode()`. Only 2 observations were removed when observations with `NA` were removed, which should not have a drastic effect on our data. 

The dataset comprises 142 newsletter subscribers and 52 non-subscribers, indicating an imbalanced class distribution.

In [ ]:
library(tidyverse)
library(tidymodels)
library(cowplot)
library(repr)

set.seed(4)
options(repr.plot.width = 14)

players_url <- "https://raw.githubusercontent.com/oo74/DSCI-100-Project/d932a95bab3bbe9a443dcba02939882b0735483f/data/players.csv"
players <- read_csv(players_url) |>
    mutate(subscribe = fct_recode(as_factor(subscribe), Yes = "TRUE", No = "FALSE"))

players |>
    map_df(n_distinct)

players |>
    summarize(mean = mean(played_hours, na.rm = TRUE),
              SD = sd(played_hours, na.rm = TRUE),
              min = min(played_hours, na.rm = TRUE),
              max = max(played_hours, na.rm = TRUE),
              median = median(played_hours, na.rm = TRUE))

players |>
    summarize(mean = mean(Age, na.rm = TRUE),
              SD = sd(Age, na.rm = TRUE),
              min = min(Age, na.rm = TRUE),
              max = max(Age, na.rm = TRUE),
              median = median(Age, na.rm = TRUE))

nrow(players)
players <- drop_na(players)
nrow(players)

players |>
    group_by(subscribe) |>
    summarize(count = n())

To avoid overplotting and help visualize density, the opacity of the datapoints was set to 0.6, with more opaque points signifying more overlap.

Fig. 1 shows that there is no obvious linear relationship between hours played and age. Most players are clustered near 0 hours of play, while a few outliers report much higher played hours (150+). The figure also indicates that the majority of players are between about 15 and 28 years old. In fact, Fig. 2 shows that a large number of players are 17 years old. Interestingly, Fig. 2 suggests that regular players have the highest average played hours, followed by amateurs, pros, beginners, and finally, veterans. I found this surprising as I assumed that more experienced players (pros and veterans) would play more. However, the outliers in Fig. 1 may explain why the regular and amateur players’ average hours are so high: several regulars have between 150–250 hours, and one amateur has around 150 hours, which greatly increases the overall average for both of these groups. Finally, Fig. 3 shows that non-binary players tend to have the highest played hours, followed by female, agender, male, those who preferred not to answer, other, and two-spirited players. The average played_hours of the three lowest gender groups are close to 0 hours.

In [ ]:
age_hours_plot <- players |>
    ggplot(aes(x = Age, y = played_hours, color = subscribe)) +
    geom_point(alpha = 0.6) +
    labs(x = "Age (years)", y = "Played Hours", title = "Fig. 1: Age, played hours, and actual subscription status of all players.", color = "Subscribed?") +
    theme(text = element_text(size = 16), legend.position = "bottom", legend.direction = "horizontal") +
    scale_color_manual(values = c("red", "chartreuse3"))

age_plot <- players |>
    ggplot(aes(x = Age, fill = subscribe)) +
    geom_histogram(binwidth = 1) +
    labs(x = "Age (years)", y = "Count", title = "Fig. 2: Distribution of age of players and their corresponding subscription status.") +
    theme(text = element_text(size = 10), legend.position = "bottom", legend.direction = "horizontal") +
    scale_fill_manual(values = c("red", "chartreuse3"))

hours_plot <- players |>
    ggplot(aes(x = played_hours, fill = subscribe)) +
    geom_histogram(binwidth = 2) +
    labs(x = "Played Hours", y = "Count", title = "Fig. 3: Distribution of players' played hours and their corresponding subscription status.") +
    theme(text = element_text(size = 10), legend.position = "bottom", legend.direction = "horizontal") +
    scale_fill_manual(values = c("red", "chartreuse3"))


age_hours_plot
plot_grid(age_plot, hours_plot, ncol = 2)

Before building the model, 75% of the data was split as the training set, and 25% was the testing set. This is important as training and testing the classifier on the same body of data could lead to misleadingly high performance metrics when, in reality, the model is simply memorizing the training data, rather than actually discovering underlying relationships needed for accurate classification of new observations.

The K values tested ranged from 1 to 25. 5-fold cross validations were performed, and the mean accuracies were found for these K's. The best K was determined as the one with the highest accuracy. If multiple K's had the same highest accuracy, the lowest K was selected for computational efficiency. 

In [ ]:
k_vals <- tibble(neighbors = 1:25)

players_split <- initial_split(players, prop = 0.75, strata = subscribe)
players_train <- training(players_split)
players_test <- testing(players_split)

players_recipe <- recipe(subscribe ~ played_hours + Age, data = players_train) |>
    step_center(all_predictors()) |>
    step_scale(all_predictors()) 

players_spec <- nearest_neighbor(weight_func = "rectangular", neighbors = tune()) |>
    set_engine("kknn") |>
    set_mode("classification")

vfold_sets <- players_train |>
    vfold_cv(v = 5, strata = subscribe)

k_accuracies <- workflow() |>
    add_recipe(players_recipe) |>
    add_model(players_spec) |>
    tune_grid(resamples = vfold_sets, grid = k_vals) |>
    collect_metrics() |>
    filter(.metric == "accuracy") |>
    mutate(accuracy = mean) |>
    select(neighbors, accuracy)
    
k_accuracies_plot <- k_accuracies |>
    ggplot(aes(x = neighbors, y = accuracy)) +
    geom_point() +
    geom_line() +
    labs(x = "Neighbors (K)", y = "Accuracy", title = "Fig. 1: Accuracies associated with various K values.") +
    theme(text = element_text(size = 12))

best_k <- k_accuracies |>
    slice_max(accuracy) |>
    slice_min(neighbors) |>
    pull(neighbors)

k_accuracies_plot

Now that the best K has been determined, we will build a new model using that K, and use it to predict for the test set.

Initial visualizations showed that most observations had low player hours (under 2.5 hours), so the maximum value displayed by the y-axis was set to 2.5 to make the majority of datapoints are more spaced out and easier to see. 

In [ ]:
players_best_spec <- nearest_neighbor(weight_func = "rectangular", neighbors = best_k) |>
    set_engine("kknn") |>
    set_mode("classification")

players_fit <- workflow() |>
    add_recipe(players_recipe) |>
    add_model(players_best_spec) |>
    fit(data = players_train)

players_predicted <- players_fit |>
    predict(players_test) |>
    bind_cols(players_test) 

players_plot <- players_predicted |>
    ggplot(aes(x = Age, y = played_hours, color = subscribe)) +
    geom_point(alpha = 0.6) +
    labs(x = "Age (years)", y = "Played Hours", title = "Fig. 2: Age, played hours, and actual subscription status of players in test set.", color = "Subscribed?") +
    ylim(0, 2.5) +
    theme(text = element_text(size = 11), legend.position = "bottom", legend.direction = "horizontal") +
    scale_color_manual(values = c("red", "chartreuse3"))

players_predicted_plot <- players_predicted |>
    ggplot(aes(x = Age, y = played_hours, color = .pred_class)) +
    geom_point(alpha = 0.6) +
    labs(x = "Age (years)", y = "Played Hours", title = "Fig. 3: Age, played hours, and predicted subscription status of players in test set", color = "Predicted to subscribe?") +
    ylim(0, 2.5) +
    theme(text = element_text(size = 11), legend.position = "bottom", legend.direction = "horizontal") +
    scale_color_manual(values = "chartreuse3")

plot_grid(players_plot, players_predicted_plot, ncol = 2)

Metrics were determined for the model, with "Yes" being the class of interest, representing "positive" in the recall and precision calculations.

The accuracy of the model is 0.7346939, the recall was 1 (perfect), and the precision was 0.7346939. Based on the confusion matrix, it appears that the model predicted every observation to be subscribed. 

In [ ]:
players_accuracy <- players_predicted |>
    metrics(truth = subscribe, estimate = .pred_class) |>
    filter(.metric == "accuracy")

players_precision <- players_predicted |>
    precision(truth = subscribe, estimate = .pred_class, event_level = "second")

players_recall <- players_predicted |>
    recall(truth = subscribe, estimate = .pred_class, event_level = "second")

players_conf_mat <- players_predicted |>
    conf_mat(truth = subscribe, estimate = .pred_class)

players_conf_mat
bind_rows(players_accuracy, players_precision, players_recall) |>
    select(-.estimator)

just because the metrix were good doesn't mean that this was a good model; model predicted yes for everything. We se this in the perfect recall, suggesting that all actual "yes" to subscribed in the data received a yes prediction. The accuracy is identitcal to precision, and this makes sense: accuracy is no. of accurate predictions over total predictions and precision is actual positive predicted as positive over the number of positive predictions. Since all predictions made were positive, and the number of accurate predictions are only those that are positive that were predicted as positive, the, respectively, numerator and denominators are same for both metrics. 
analyze accuracy, precision, and recall

futhermore, tends to predict subscribed because of class imbalance. - high recall, so minimizes false positives. Thus, upsampling should be performed during pre-processing to minimize the impact of unbalanced classes on classification. This is also why the accuracy remaains relatively high at 0.7346939 in spite of the model always predicting positive––there were not a lot of negatives, so the few false positive classifications did not affect the accuracy as much. Still an accuracy of 0.7346939 is not completely high––roughly 3 in every 10 individuals will be misclassified when using the non-upsampled predictors age and player hours. It is unsure whether these predictors are inherently bad predictors of subscription status, or if the subpar performance more so resulted from the unbalanced class, but either way, the model is not good. 

## (4) Discussion:
- summarize what you found
- discuss whether this is what you expected to find?
- discuss what impact could such findings have?
- discuss what future questions could this lead to?